# dim_region — SCD Type 1 (SIMPLE PySpark)

**No SQL MERGE.** Uses `DeltaTable.merge()` Python API only.
Three steps: read silver → upsert → done.

In [0]:
# ============================================================
#  dim_region  —  SCD Type 1  (SIMPLE PySpark, NO SQL MERGE)
# ============================================================
#  SCD1 = "overwrite on key" — we keep ONE row per region_id and
#  update it in place when business attributes change.
#
#  3 steps:
#    1. Read latest snapshot from silver (one row per region_id).
#    2. Use DeltaTable.merge() (Python API) — NOT SQL MERGE — to
#       UPSERT: insert new keys, update existing ones.
#    3. Done.
# ============================================================
from pyspark.sql import functions as F, Window as W
from delta.tables import DeltaTable

dbutils.widgets.dropdown(name="environment", defaultValue="dev",
                         choices=["dev","qa","prd"], label="select Environment")
env = dbutils.widgets.get("environment")

silver_tbl = f"saleslake_{env}.silver_{env}.cleanedRegion"
gold_tbl   = f"saleslake_{env}.gold_{env}.dim_region"

# ---------- 1. Latest snapshot from silver ----------
src = (spark.table(silver_tbl)
            .withColumn("rn", F.row_number().over(
                  W.partitionBy("region_id").orderBy(F.col("ingest_ts").desc())))
            .filter("rn = 1")
            .drop("rn", "ingest_ts"))

# ---------- 2. Upsert via Delta Python API (DeltaTable.merge) ----------
gold = DeltaTable.forName(spark, gold_tbl)

(gold.alias("tgt")
     .merge(src.alias("src"),
            "tgt.region_id = src.region_id")
     .whenMatchedUpdate(set={
         "region_code":      "src.region_code",
         "region_name":      "src.region_name",
         "country":          "src.country",
         "sub_region":       "src.sub_region",
         "regional_manager": "src.regional_manager",
         "last_updt_ts":     "current_timestamp()",
     })
     .whenNotMatchedInsert(values={
         "region_id":        "src.region_id",
         "region_code":      "src.region_code",
         "region_name":      "src.region_name",
         "country":          "src.country",
         "sub_region":       "src.sub_region",
         "regional_manager": "src.regional_manager",
         "initial_load_ts":  "current_timestamp()",
         "last_updt_ts":     "current_timestamp()",
     })
     .execute())

print(f"dim_region SCD1 load complete")
spark.table(gold_tbl).display()

